# ¿Cuando el mercado tiene pánico, el Bitcoin sube o cae?
## Lag correlation + Event Study + Rocket & Feather sobre BTC y Fear & Greed

**Dilema:** El Fear & Greed Index mide el sentimiento del mercado crypto (0=pánico, 100=euforia).
La teoría contraria dice "compra cuando todos tienen miedo". ¿Los datos lo confirman?
¿O el pánico simplemente acompaña a las caídas sin predecirlas?

**Métodos:**
1. EDA de las dos series
2. Lag correlation: ¿F&G predice BTC a T+k días?
3. Event study: ±15 días cuando Fear & Greed < 20 (Extreme Fear)
4. Rocket & Feather: ¿el pánico cae más rápido que la confianza se recupera?

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d0d0d',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#333',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'text.color':       '#eee',
    'grid.color':       '#2a2a2a',
    'grid.linewidth':   0.6,
    'font.family':      'monospace',
    'axes.titlesize':   12,
})

PINK   = '#f4a7b9'
BLUE   = '#7eb8f7'
GREEN  = '#9ece6a'
ORANGE = '#e0af68'
SEED   = 42
np.random.seed(SEED)
print('Setup OK')

## 1. Datos

In [ ]:
DATA_DIR = Path('../data')

def load_real_data():
    """Carga datos reales si están disponibles."""
    merged = DATA_DIR / 'btc_fng_merged.csv'
    if merged.exists():
        df = pd.read_csv(merged, parse_dates=['date'])
        print(f'Datos reales cargados: {len(df)} días')
        return df
    # Intentar cargar por separado
    btc_f = DATA_DIR / 'btc_price.csv'
    fng_f = DATA_DIR / 'fear_greed.csv'
    if btc_f.exists() and fng_f.exists():
        btc = pd.read_csv(btc_f, parse_dates=['date'])
        fng = pd.read_csv(fng_f, parse_dates=['date'])
        df = pd.merge(btc, fng, on='date', how='inner')
        print(f'Datos reales cargados y mergeados: {len(df)} días')
        return df
    return None


def generate_synthetic():
    """
    Simula BTC price + Fear & Greed Index con comportamiento realista.
    - BTC: random walk con drift positivo y shocks de volatilidad
    - F&G: correlacionado con BTC pero con retardo y ruido propio
    - Efecto contrario embedded: en Extreme Fear, rebote moderado T+7
    """
    dates = pd.date_range('2022-01-01', '2024-12-31', freq='D')
    n = len(dates)

    # Precio BTC: random walk log-normal
    log_returns = np.random.normal(0.0003, 0.035, n)

    # Añadir shocks: crashes y rallies conocidos
    shocks = {
        # (fecha, duración días, magnitud)
        '2022-05-10':  (14, -0.55),   # Luna/UST crash
        '2022-11-08':  (10, -0.28),   # FTX collapse
        '2023-01-16':  (20, +0.40),   # rally enero
        '2023-10-15':  (30, +0.35),   # pre-ETF anticipación
        '2024-03-05':  (15, +0.25),   # ATH marzo
        '2024-08-05':  (7,  -0.22),   # macro sell-off
    }

    for date_str, (dur, mag) in shocks.items():
        try:
            idx = dates.get_loc(pd.Timestamp(date_str))
            shock_impact = mag / dur
            for d in range(min(dur, n - idx)):
                log_returns[idx + d] += shock_impact
        except Exception:
            pass

    btc_price = 40000 * np.exp(np.cumsum(log_returns))
    btc_price = np.clip(btc_price, 10000, 120000)

    # Fear & Greed: función del retorno de BTC (con lag y ruido)
    ret_7d = pd.Series(btc_price).pct_change(7).fillna(0).values
    fng_raw = 50 + 300 * ret_7d + np.random.normal(0, 12, n)
    fng = np.clip(fng_raw, 0, 100).astype(int)

    df = pd.DataFrame({
        'date':      dates,
        'price_usd': np.round(btc_price, 2),
        'fng':       fng,
        'label':     pd.cut(fng,
                            bins=[0, 25, 45, 55, 75, 100],
                            labels=['Extreme Fear','Fear','Neutral','Greed','Extreme Greed'],
                            include_lowest=True),
    })

    print(f'Dataset sintético: {len(df)} días ({df["date"].dt.year.min()}–{df["date"].dt.year.max()})')
    return df


df = load_real_data()
if df is None:
    print('Sin datos reales — generando dataset sintético')
    df = generate_synthetic()

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Retornos
df['ret_1d']  = df['price_usd'].pct_change(1)
df['ret_7d']  = df['price_usd'].pct_change(7)
df['ret_14d'] = df['price_usd'].pct_change(14)

print(df[['date','price_usd','fng']].tail())

## 2. EDA — Las dos series

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Bitcoin & Fear & Greed Index', fontsize=14)

# BTC precio
ax1.plot(df['date'], df['price_usd'], color=ORANGE, lw=1.5)
ax1.set_ylabel('BTC USD')
ax1.set_title('Precio Bitcoin')
ax1.grid()

# F&G coloreado por zona
fng_colors = df['fng'].apply(
    lambda x: PINK if x < 25 else (BLUE if x < 45 else (GREEN if x < 75 else ORANGE))
)
ax2.bar(df['date'], df['fng'], color=fng_colors, alpha=0.7, width=1)
ax2.axhline(25, color=PINK, lw=0.8, ls='--', label='Extreme Fear (<25)')
ax2.axhline(75, color=ORANGE, lw=0.8, ls='--', label='Extreme Greed (>75)')
ax2.set_ylabel('Fear & Greed')
ax2.set_title('Fear & Greed Index (0=pánico, 100=euforia)')
ax2.set_ylim(0, 100)
ax2.legend(fontsize=8)
ax2.grid()

# Retorno 7d
ret7 = df['ret_7d'].dropna()
colors_ret = [PINK if r < 0 else GREEN for r in ret7]
ax3.bar(df.loc[ret7.index, 'date'], ret7.values * 100, color=colors_ret, alpha=0.7, width=1)
ax3.set_ylabel('Retorno 7d (%)')
ax3.set_title('Retorno semanal BTC')
ax3.axhline(0, color='white', lw=0.8)
ax3.grid()

plt.tight_layout()
plt.savefig('../data/img_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Días con Extreme Fear (<25): {(df["fng"] < 25).sum()}')
print(f'Días con Extreme Greed (>75): {(df["fng"] > 75).sum()}')

## 3. Lag Correlation — ¿F&G predice BTC a T+k días?

In [ ]:
MAX_LAG = 21  # 3 semanas
clean = df[['fng', 'price_usd']].dropna()

lag_results = []
for lag in range(-MAX_LAG, MAX_LAG + 1):
    x = clean['fng']
    y = clean['price_usd'].shift(-lag)  # lag negativo = F&G predice BTC futuro
    valid = pd.concat([x, y], axis=1).dropna()
    if len(valid) < 50:
        continue
    r, p = stats.pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
    lag_results.append({'lag': lag, 'r': r, 'p': p, 'sig': p < 0.05})

lag_df = pd.DataFrame(lag_results)
best = lag_df.loc[lag_df['r'].abs().idxmax()]

print(f'Máxima correlación: lag = {int(best["lag"])} días, r = {best["r"]:+.4f}')
print(f'  lag > 0: F&G de hace {int(best["lag"])} días predice el precio de hoy')
print(f'  lag < 0: F&G de hoy predice el precio en {-int(best["lag"])} días')

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Lag Correlation — Fear & Greed vs precio BTC')

colors = [PINK if row['r'] < 0 else BLUE for _, row in lag_df.iterrows()]
ax.bar(lag_df['lag'], lag_df['r'], color=colors, alpha=0.85)

# Significativo: borde blanco
sig = lag_df[lag_df['sig']]
ax.bar(sig['lag'], sig['r'], color=[PINK if r < 0 else GREEN for r in sig['r']],
       alpha=1.0, edgecolor='white', linewidth=0.8)

ax.axvline(0, color='white', lw=1, ls='--')
ax.axhline(0, color='#555', lw=0.8)
ax.set_xlabel('lag (días)\n← F&G predice pasado | F&G predice futuro →')
ax.set_ylabel('Pearson r')
ax.set_xlim(-MAX_LAG - 0.5, MAX_LAG + 0.5)

legend = [
    mpatches.Patch(color=GREEN, label='Correlación positiva (significativa)'),
    mpatches.Patch(color=PINK,  label='Correlación negativa'),
    mpatches.Patch(color=BLUE,  label='Positiva (no significativa)'),
]
ax.legend(handles=legend, fontsize=8)
ax.grid(axis='y')

plt.tight_layout()
plt.savefig('../data/img_lag_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Event Study — ±15 días alrededor de Extreme Fear

In [ ]:
WINDOW = 15
EXTREME_FEAR = 20

# Identificar eventos de Extreme Fear (evitar solapamiento: mín 30 días entre eventos)
ef_dates = df[df['fng'] <= EXTREME_FEAR]['date'].values
events = []
last = pd.Timestamp('1900-01-01')
for d in ef_dates:
    d = pd.Timestamp(d)
    if (d - last).days >= 30:
        events.append(d)
        last = d

print(f'Eventos Extreme Fear (F&G ≤ {EXTREME_FEAR}, ventana mín 30d): {len(events)}')
for e in events:
    print(f'  {e.date()}')

# Construir panel de eventos
event_panel = []
for event in events:
    event_price = df[df['date'] == event]['price_usd'].values
    if len(event_price) == 0:
        continue
    base = event_price[0]
    for t in range(-WINDOW, WINDOW + 1):
        target_date = event + pd.Timedelta(days=t)
        row = df[df['date'] == target_date]
        if len(row) == 0:
            continue
        event_panel.append({
            'event_date': event,
            't':          t,
            'car':        (row['price_usd'].values[0] / base - 1) * 100,
        })

ep = pd.DataFrame(event_panel)
agg = ep.groupby('t')['car'].agg(['mean', 'std', 'count']).reset_index()
agg['se'] = agg['std'] / np.sqrt(agg['count'])

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.set_title(f'Event Study — ±{WINDOW} días alrededor de Extreme Fear (F&G ≤ {EXTREME_FEAR})\n'
             f'n={len(events)} eventos  |  precio BTC relativo al día del evento')

ax.plot(agg['t'], agg['mean'], color=ORANGE, lw=2.5, marker='o', ms=4)
ax.fill_between(agg['t'],
                agg['mean'] - 1.96*agg['se'],
                agg['mean'] + 1.96*agg['se'],
                color=ORANGE, alpha=0.15, label='IC 95%')

ax.axvline(0,  color='white',   lw=1.5, ls='--', label='día del Extreme Fear')
ax.axhline(0,  color='#555',    lw=0.8)
ax.fill_between(agg[agg['t'] >= 0]['t'], agg[agg['t'] >= 0]['mean'], 0,
                alpha=0.07, color=GREEN if agg[agg['t'] >= 1]['mean'].mean() > 0 else PINK)

ax.set_xlabel('días respecto al Extreme Fear')
ax.set_ylabel('retorno acumulado % (base=día 0)')
ax.legend(fontsize=9)
ax.grid()

# Anotar retorno en T+7 y T+15
for t in [7, 14]:
    val = agg[agg['t'] == t]['mean'].values
    if len(val):
        ax.annotate(f'T+{t}: {val[0]:+.1f}%',
                    xy=(t, val[0]), xytext=(t+1, val[0]+1.5),
                    fontsize=9, color='white',
                    arrowprops=dict(arrowstyle='->', color='white', lw=0.8))

plt.tight_layout()
plt.savefig('../data/img_event_study.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Rocket & Feather — ¿El pánico cae más rápido que la confianza se recupera?

In [ ]:
"""
Rocket & Feather adaptado a sentimiento:
  - 'Rocket': ¿Cuántos días tarda el F&G en caer ≥20 puntos desde un pico?
  - 'Feather': ¿Cuántos días tarda en recuperar ≥20 puntos desde un mínimo?
"""

fng_series = df['fng'].values
dates_arr  = df['date'].values

falls     = []  # días para caer 20 puntos
recoveries = []  # días para subir 20 puntos
DELTA = 20

for i in range(len(fng_series) - 30):
    v0 = fng_series[i]
    # Caída: buscar cuándo bajó DELTA desde aquí
    if v0 >= 40:  # solo desde zona neutral/positiva
        for j in range(i+1, min(i+60, len(fng_series))):
            if fng_series[j] <= v0 - DELTA:
                falls.append(j - i)
                break
    # Recuperación: buscar cuándo subió DELTA desde aquí
    if v0 <= 40:  # solo desde zona negativa/neutral
        for j in range(i+1, min(i+60, len(fng_series))):
            if fng_series[j] >= v0 + DELTA:
                recoveries.append(j - i)
                break

falls      = np.array(falls)
recoveries = np.array(recoveries)

if len(falls) > 5 and len(recoveries) > 5:
    stat, p_rk = stats.mannwhitneyu(falls, recoveries, alternative='less')
    print(f'Días para CAER {DELTA} puntos:      media={falls.mean():.1f}, mediana={np.median(falls):.0f}')
    print(f'Días para RECUPERAR {DELTA} puntos: media={recoveries.mean():.1f}, mediana={np.median(recoveries):.0f}')
    print(f'Mann-Whitney: p={p_rk:.4f}')
    print(f'Conclusión: el pánico {"cae MÁS RÁPIDO" if p_rk < 0.05 else "no cae significativamente más rápido"} que la confianza se recupera')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Rocket & Feather — Sentimiento crypto\n'
                 f'¿El pánico cae más rápido de lo que la confianza se recupera? (Mann-Whitney p={p_rk:.3f})')

    axes[0].hist(falls,      bins=20, color=PINK,  alpha=0.85, label=f'Caída {DELTA}pts (n={len(falls)})')
    axes[0].hist(recoveries, bins=20, color=GREEN, alpha=0.65, label=f'Recuperación {DELTA}pts (n={len(recoveries)})')
    axes[0].axvline(falls.mean(),      color=PINK,  lw=2, ls='--')
    axes[0].axvline(recoveries.mean(), color=GREEN, lw=2, ls='--')
    axes[0].set_xlabel('días')
    axes[0].set_ylabel('frecuencia')
    axes[0].set_title('Distribución de velocidades')
    axes[0].legend(fontsize=8)
    axes[0].grid(axis='y')

    axes[1].boxplot([falls, recoveries],
                    labels=[f'Caída\n{DELTA}pts', f'Recuperación\n{DELTA}pts'],
                    patch_artist=True,
                    boxprops=dict(facecolor='#1a1a1a', color='white'),
                    medianprops=dict(color='white', lw=2),
                    whiskerprops=dict(color='white'),
                    capprops=dict(color='white'),
                    flierprops=dict(marker='o', color='#555', ms=4))
    for i, (data, color) in enumerate([(falls, PINK), (recoveries, GREEN)], 1):
        axes[1].scatter(np.random.normal(i, 0.06, min(len(data), 200)),
                        np.random.choice(data, min(len(data), 200)),
                        color=color, alpha=0.3, s=15, zorder=3)
    axes[1].set_ylabel('días')
    axes[1].set_title('Boxplot comparativo')
    axes[1].grid(axis='y')

    plt.tight_layout()
    plt.savefig('../data/img_rocket_feather.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Insuficientes eventos para el análisis Rocket & Feather.')
    print(f'  Caídas encontradas: {len(falls)}, Recuperaciones: {len(recoveries)}')

## 6. Conclusiones

In [ ]:
print('=== RESUMEN ===')
print()
print('Pregunta: ¿cuando el mercado tiene pánico, el Bitcoin sube o cae?')
print()

best_fwd = lag_df[lag_df['lag'] < 0].loc[lag_df[lag_df['lag'] < 0]['r'].abs().idxmax()]
print(f'1. Lag correlation:')
print(f'   F&G de hoy predice BTC en {int(-best_fwd["lag"])} días: r = {best_fwd["r"]:+.3f}')
if best_fwd['r'] > 0:
    print('   → F&G alto (euforia) se asocia con precios más altos T+k días')
else:
    print('   → F&G alto (euforia) se asocia con precios más bajos T+k días (señal contraria)')

print()
t7 = agg[agg['t'] == 7]['mean'].values[0] if len(agg[agg['t'] == 7]) else 0
print(f'2. Event Study (Extreme Fear):')
print(f'   7 días después del Extreme Fear: retorno medio = {t7:+.1f}%')
if t7 > 0:
    print('   → Confirma la teoría contraria: el pánico extremo precede a rebotes')
else:
    print('   → No confirma la teoría contraria: el pánico acompaña caídas posteriores')

if len(falls) > 5 and len(recoveries) > 5:
    print()
    print(f'3. Rocket & Feather:')
    print(f'   Caída {DELTA}pts: media={falls.mean():.1f}d | Recuperación {DELTA}pts: media={recoveries.mean():.1f}d')
    if falls.mean() < recoveries.mean():
        print('   → El sentimiento cae más rápido de lo que se recupera (asimetría confirmada)')
    else:
        print('   → El sentimiento se recupera más rápido de lo que cae')